# Modélisation – H5 : le volume et l'élan annoncent le sens de l'évolution
**Contributrice :** Gabrielle · **Date :** 10/11/2026

Notebook construit à partir du template commun de l'équipe. Il passe par `commun.py` pour tout ce qui doit être identique d'une hypothèse à l'autre (la table, le masque des lignes, les fenêtres, la naïve, `evaluer` et `tester`), et par `gabarit_pont.py` pour le modèle pont, base de toutes les hypothèses du tour 2.

**Question :** par territoire, le volume de ventes de N-1, son évolution et l'élan du prix améliorent-ils la prévision du prix N ?

**Protocole (feuille de route, tour 2)**
- **Avant de modéliser, une analyse préalable** sur les années d'apprentissage seulement (prix 2017 à 2022) : chaque territoire × année est classé en hausse, stable ou baisse selon l'évolution du prix entre N-1 et N ; on teste si chaque indicateur de N-1 sépare les trois groupes (Kruskal-Wallis et taille d'effet). Cette analyse fixe aussi la **bande « stable »** pour toute l'équipe (Fantôme la réutilise pour H6).
- **Modèles :** le modèle pont + les indicateurs qui séparent les groupes, ajoutés un par un. L'année 2020 (confinement) : indicatrice testée en sensibilité, et limite déclarée.
- **Ce qu'on regarde :** la règle de décision contre le modèle pont, avec l'année 2023 lue à part.

Les indicateurs viennent de P9 (`P9_volume_elan.ipynb`, fichier `p9_volume_elan.parquet`). Le test 2025 n'est pas ouvert.

## 0. Paramètres de l'hypothèse
- `HYPOTHESE` : préfixe du nom des modèles dans les tableaux.
- `INDICATEURS` : les candidats de l'analyse préalable. Le volume de N-1 (`nb_ventes_n1`, déjà dans la table) et les quatre indicateurs de P9.
- `LOG_H5` : les indicateurs asymétriques passés en log dans le modèle pont (règle de l'équipe : asymétrie > 1). `nb_ventes_n1` est dans la liste `LOG` de l'équipe. L'évolution des ventes (asymétrie 1,1, avec des valeurs négatives) entre sous la forme du rapport ventes N-1 / ventes N-2, passé en log : c'est la même information, rendue symétrique.
- `SEUIL_EPSILON2` : la taille d'effet minimale pour dire qu'un indicateur sépare les groupes, **fixée avant les résultats** : 0,01, le seuil conventionnel d'un « petit » effet.

In [1]:
HYPOTHESE = "H5"

INDICATEURS = ["nb_ventes_n1", "rotation_parc", "rotation_parc_type", "evol_ventes_n2_n1", "evol_prix_n2_n1"]
COLONNES_P9 = ["rotation_parc", "rotation_parc_type", "evol_ventes_n2_n1", "evol_prix_n2_n1"]

# Forme donnée au modèle pour chaque indicateur, et indicateurs passés en log dans le pont
FORME_MODELE = {"nb_ventes_n1": "nb_ventes_n1", "rotation_parc": "rotation_parc",
                "rotation_parc_type": "rotation_parc_type", "evol_ventes_n2_n1": "rapport_ventes_n2_n1",
                "evol_prix_n2_n1": "evol_prix_n2_n1"}
LOG_H5 = ["nb_ventes_n1", "rapport_ventes_n2_n1"]

SEUIL_EPSILON2 = 0.01       # « petit » effet : fixé avant les résultats
SEUIL_GAIN = 3              # % de gain sur la MAE de la comparaison (règle de l'équipe)
ANNEE_DEBUT = 2017          # période longue (P6, décision H4)

## 1. Import et chargement
`charger_table()` applique le masque commun. On y joint les indicateurs de P9 sur les clés (année, sous-bassin, type de bien), sans toucher à la table de l'équipe.

**Lignes de H5.** Les deux évolutions de P9 sont vides quand le territoire avait moins de 20 ventes en N-2 (jusqu'à 7,5 % des lignes d'appartements selon l'année, P9 section 3). Le modèle pont ne prend pas de case vide. On garde donc les seules lignes où les indicateurs sont renseignés, **pour tous les modèles de ce notebook, pont compris** : la comparaison au pont se fait sur les mêmes lignes. Les MAE du pont diffèrent donc un peu de celles publiées dans H4 (tableau de contrôle ci-dessous).

In [2]:
import warnings
import numpy as np
import pandas as pd
from scipy import stats

import commun
from commun import CIBLE, charger_table, fenetres, evaluer, tester
from gabarit_pont import ModelePont, previsions_pont, MODELE_COMPLET

warnings.filterwarnings("ignore")      # convergence de l'Elastic Net sur les plis les plus petits

table_commune = charger_table()

# Jointure de P9 sur les clés, sans dépendre de la forme de l'index
p9 = pd.read_parquet(commun.DATA_OUT / "p9_volume_elan.parquet")
cles = pd.DataFrame({"annee": table_commune["annee"].to_numpy(),
                     "sous_bassin_de_vie": table_commune.index.get_level_values("sous_bassin_de_vie"),
                     "type_bien": table_commune.index.get_level_values("type_bien")})
joint = cles.merge(p9, on=["annee", "sous_bassin_de_vie", "type_bien"], how="left", validate="one_to_one")
assert len(joint) == len(table_commune)
table_p9 = table_commune.copy()
for c in COLONNES_P9:
    table_p9[c] = joint[c].to_numpy()
table_p9["rapport_ventes_n2_n1"] = 1 + table_p9["evol_ventes_n2_n1"] / 100

# Lignes de H5 : indicateurs renseignés
complet = table_p9[COLONNES_P9].notna().all(axis=1)
table = table_p9[complet]
retirees = (~complet).groupby([table_p9["annee"], table_p9.index.get_level_values("type_bien")]).sum().unstack()
print("Table commune :", table_commune.shape, "| lignes de H5 :", table.shape)
print("Lignes retirées (indicateur vide), par année :")
retirees.loc[ANNEE_DEBUT:]

Table commune : (23489, 39) | lignes de H5 : (23177, 44)
Lignes retirées (indicateur vide), par année :


type_bien,appartement,maison
annee,,
2017,40,2
2018,42,5
2019,37,1
2020,64,1
2021,32,1
2022,50,1
2023,11,1
2024,3,0
2025,19,2


In [3]:
FENETRES = fenetres(table, annee_debut=ANNEE_DEBUT)
for a, (app, ev) in FENETRES.items():
    print(f"Fenêtre {a} : apprentissage {sorted(int(x) for x in app['annee'].unique())} ({len(app)} lignes) | jugée {a} ({len(ev)} lignes)")

Fenêtre 2023 : apprentissage [2017, 2018, 2019, 2020, 2021, 2022] (15369 lignes) | jugée 2023 (2629 lignes)
Fenêtre 2024 : apprentissage [2017, 2018, 2019, 2020, 2021, 2022, 2023] (17998 lignes) | jugée 2024 (2596 lignes)


## 2. Analyse préalable : le volume et l'élan séparent-ils hausse, stable et baisse ?
Sur les années d'apprentissage de la première fenêtre seulement (prix 2017 à 2022) : 2023 et 2024 ne sont jamais regardées avant la modélisation.

### 2.1 La bande « stable »
L'évolution observée d'un territoire, prix / prix N-1 − 1, mélange un vrai mouvement et le **bruit des médianes** : avec peu de ventes, la médiane bouge beaucoup d'une année sur l'autre sans que le marché ait bougé. La bande « stable » est l'intervalle autour de 0 en deçà duquel on ne parle pas de mouvement. Deux options sont sur la table :
- **fixe** : la même bande pour tous les territoires ;
- **dépendante du nombre de ventes** : plus large là où le bruit est grand.

**Mesure du bruit, années calmes 2017-2019.** L'écart de chaque territoire à la médiane de son année (le marché commun retiré) suit très bien la loi du bruit d'une différence de deux médianes : écart-type ≈ c × √(1/ventes N + 1/ventes N-1). On estime c par type de bien (écart absolu médian × 1,4826, estimateur robuste de l'écart-type). Les ventes de N ne servent qu'à classer, jamais de variable.

In [5]:
app = FENETRES[commun.ANNEES_JUGEES[0]][0].copy()          # prix 2017 à 2022
app["type"] = app.index.get_level_values("type_bien")   # copie en colonne (type_bien reste dans l'index)
app["evolution"] = (app[CIBLE] / app["prix_n1"] - 1) * 100
app["inverse_ventes"] = np.sqrt(1 / app["n_ventes"] + 1 / app["nb_ventes_n1"])

calme = app[app["annee"] <= 2019]
ecart_annee = calme["evolution"] - calme.groupby([calme["annee"], calme["type"]])["evolution"].transform("median")
C_BRUIT = {tb: 1.4826 * float((ecart_annee[g.index].abs() / g["inverse_ventes"]).median())
           for tb, g in calme.groupby("type")}
print("c par type de bien :", {k: round(v, 1) for k, v in C_BRUIT.items()})

# Contrôle de la loi du bruit : écart-type robuste observé et prévu, par quintile de taille du marché
taille = pd.qcut(calme["inverse_ventes"].rank(method="first"), 5, labels=["Q1 (grands)", "Q2", "Q3", "Q4", "Q5 (petits)"])
controle_bruit = pd.DataFrame({
    "observé (%)": (1.4826 * ecart_annee.abs()).groupby([calme["type"], taille]).median(),
    "prévu (%)": (calme["type"].map(C_BRUIT) * calme["inverse_ventes"]).groupby([calme["type"], taille]).median(),
}).round(1)
controle_bruit.unstack(0)

c par type de bien : {'appartement': 46.5, 'maison': 50.2}


observé (%)          prévu (%)       
type           appartement maison appartement maison
inverse_ventes                                      
Q1 (grands)            3.6    3.1         2.5    3.2
Q2                     3.8    4.6         4.1    4.5
Q3                     5.4    5.4         5.4    5.8
Q4                     6.2    7.3         7.0    7.3
Q5 (petits)            9.1   10.8        10.4    9.7

**Le critère de choix, fixé avant de regarder les tests.** Une bonne bande classe les territoires selon leur marché, pas selon leur taille. Sinon, n'importe quel indicateur lié à la taille (le volume de ventes, la rotation) séparerait les groupes mécaniquement, et H5 testerait la construction des classes au lieu du marché. On mesure donc, pour chaque bande, le lien entre la classe et le quintile de taille du marché (V de Cramér, de 0 : aucun lien, à 1). **La bande retenue est celle dont le classement dépend le moins de la taille.**

- Bandes fixes de ±1 % à ±8 %.
- Bandes dépendantes du nombre de ventes : ±z écarts-types de bruit, z de 0,25 à 1,5.

In [6]:
def classer(evolution, bande):
    """Classe une évolution (en %) : 'hausse' au-dessus de +bande, 'baisse' sous −bande, 'stable' entre les deux."""
    return pd.Series(np.select([evolution > bande, evolution < -bande], ["hausse", "baisse"], "stable"),
                     index=evolution.index)


def v_cramer(classes, e):
    """V de Cramér entre la classe et le quintile de taille du marché (ventes N-1)."""
    q = pd.qcut(e["nb_ventes_n1"].rank(method="first"), 5)
    ct = pd.crosstab(q, classes.loc[e.index])
    return np.sqrt(stats.chi2_contingency(ct)[0] / (len(e) * (min(ct.shape) - 1)))


bruit = app["type"].map(C_BRUIT) * app["inverse_ventes"]
candidates = {f"fixe ±{b} %": pd.Series(float(b), index=app.index) for b in [1, 2, 3, 4, 5, 6, 8]}
candidates |= {f"dépendante, z = {z}": z * bruit for z in [0.25, 0.5, 0.75, 1, 1.5]}

comparaison = []
for nom, bande in candidates.items():
    cl = classer(app["evolution"], bande)
    parts = cl.value_counts(normalize=True).mul(100)
    comparaison.append({"bande": nom, "largeur médiane (%)": bande.median(),
                        **{f"V de Cramér ({tb})": v_cramer(cl, e) for tb, e in app.groupby("type")},
                        **{f"{k} (%)": parts.get(k, 0) for k in ["baisse", "stable", "hausse"]}})
comparaison = pd.DataFrame(comparaison).set_index("bande")
comparaison.round(3)

,largeur médiane (%),V de Cramér (appartement),V de Cramér (maison),baisse (%),stable (%),hausse (%)
bande,,,,,,
fixe ±1 %,1.000,0.165,0.164,19.129,9.545,71.325
fixe ±2 %,2.000,0.177,0.172,15.616,18.680,65.704
fixe ±3 %,3.000,0.177,0.175,12.538,28.030,59.431
fixe ±4 %,4.000,0.173,0.174,10.332,36.307,53.361
fixe ±5 %,5.000,0.169,0.174,8.328,44.343,47.329
fixe ±6 %,6.000,0.169,0.170,6.676,52.281,41.044
fixe ±8 %,8.000,0.164,0.164,4.366,65.853,29.781
"dépendante, z = 0.25",1.364,0.173,0.161,17.203,13.475,69.321
"dépendante, z = 0.5",2.727,0.205,0.176,12.460,25.955,61.585


**Lecture**
- **La loi du bruit tient** (tableau précédent) : l'écart-type observé passe de 3 à 4 % dans les grands marchés à 9 à 11 % dans les petits, comme prévu par c × √(1/ventes N + 1/ventes N-1). Seuls les plus grands marchés d'appartements sont un peu sous-estimés (2,5 % prévu, 3,6 % observé).
- **Avec une bande fixe, la dépendance à la taille ne bouge presque pas** : V de Cramér entre 0,16 et 0,18 quelle que soit la largeur, de ±1 % à ±8 %.
- **Avec une bande dépendante du nombre de ventes, elle augmente avec la largeur** : de 0,17 (z = 0,25) à 0,30 (z = 1,5) pour les appartements, de 0,16 à 0,21 pour les maisons. À largeur médiane comparable (environ 5,5 %), 0,25 contre 0,17 pour les appartements.
- **Pourquoi** : pendant la hausse de 2020-2022, la bande étroite des grands marchés les classe presque tous en « hausse », tandis que la bande large des petits marchés les laisse « stables » pour la même évolution. La classe dit alors la taille du marché autant que son sens.
- **Conséquence** : contrairement à ce que je proposais avant l'analyse, la bande dépendante du nombre de ventes n'est pas retenue. Elle ferait séparer les groupes par les indicateurs de volume, mécaniquement. **La bande retenue est fixe.**

**La valeur de la bande fixe.** Parmi les bandes fixes, la dépendance à la taille change peu d'une largeur à l'autre : la largeur se choisit donc sur le bruit. On prend le bruit des **grands marchés** en années calmes (premier quintile de taille, écart-type robuste) : en dessous, même un grand marché ne permet pas de distinguer un mouvement de l'aléa des médianes. Arrondi au point.

In [7]:
grands = taille == "Q1 (grands)"
BANDE_STABLE = float(round(1.4826 * ecart_annee[grands].abs().median()))
print(f"Bruit des grands marchés (2017-2019) : {1.4826 * ecart_annee[grands].abs().median():.2f} % -> BANDE_STABLE = ±{BANDE_STABLE:.0f} %")

app["classe"] = classer(app["evolution"], BANDE_STABLE)
repartition = pd.crosstab([app["annee"]], [app["type"], app["classe"]], normalize="index").mul(100).round(0)
repartition

Bruit des grands marchés (2017-2019) : 3.18 % -> BANDE_STABLE = ±3 %


type   appartement               maison              
classe      baisse hausse stable baisse hausse stable
annee                                                
2017           6.0   13.0   10.0   14.0   29.0   29.0
2018           7.0   11.0   11.0   13.0   30.0   28.0
2019           5.0   15.0   10.0   13.0   30.0   26.0
2020           3.0   20.0    8.0    7.0   43.0   19.0
2021           2.0   25.0    5.0    3.0   54.0   10.0
2022           2.0   26.0    5.0    2.0   58.0    8.0

**Lecture**
- **Bande « stable » : ±3 %**, le bruit des grands marchés en années calmes (3,2 %). Sur les prix 2017 à 2022 : 12,5 % de baisses, 28 % de stables, 59 % de hausses (tableau 2.1).
- **Les classes suivent la conjoncture** : en 2017-2019, environ un tiers de stables ; en 2021-2022, plus des trois quarts des territoires en hausse et très peu de baisses. Le classement mélange donc le « quand » (l'année) et le « où » (le territoire), d'où les deux lectures de la section 2.2.
- **Pour Fantôme (H6)** : `BANDE_STABLE = 3` et la fonction `classer(evolution, bande)` de la section 2.1. Une bande fixe se recalcule sans le nombre de ventes de N, qui n'est pas connu au moment de prévoir.

### 2.2 Chaque indicateur sépare-t-il les trois groupes ?
Test de Kruskal-Wallis par indicateur et par type de bien, taille d'effet ε² = (H − k + 1) / (n − k), avec k = 3 groupes. Repères : 0,01 petit effet, 0,08 moyen, 0,26 grand.

Deux lectures, pour séparer le « quand » du « où » (leçon de H3c) :
- **brute** : l'indicateur tel quel. Il peut séparer les groupes simplement parce que les années de forte hausse (2021-2022) ont aussi des ventes et un élan élevés : c'est la part commune de l'année ;
- **écart à l'année** : l'indicateur moins sa médiane de l'année (même type de bien). Ce qui reste sépare les territoires à année égale : c'est la part propre.

Un même territoire revient chaque année : les observations ne sont pas indépendantes, les p-values sont trop optimistes. On décide sur la taille d'effet, pas sur la p-value.

In [8]:
def epsilon2(groupes):
    h, p = stats.kruskal(*groupes)
    n = sum(len(g) for g in groupes)
    return (h - len(groupes) + 1) / (n - len(groupes)), h, p


ORDRE = ["baisse", "stable", "hausse"]
lignes_kw = []
for tb, e in app.groupby("type"):
    for v in INDICATEURS:
        x = e[v]
        ecart = x - x.groupby(e["annee"]).transform("median")
        eps, h, p = epsilon2([x[e["classe"] == k] for k in ORDRE])
        eps_loc, _, _ = epsilon2([ecart[e["classe"] == k] for k in ORDRE])
        lignes_kw.append({"type de bien": tb, "indicateur": v,
                          **{f"médiane {k}": x[e["classe"] == k].median() for k in ORDRE},
                          "H": h, "p-value": p, "ε² brut": eps, "ε² écart à l'année": eps_loc})
kw = pd.DataFrame(lignes_kw)
kw.round(3)

,type de bien,indicateur,médiane baisse,médiane stable,médiane hausse,H,p-value,ε² brut,ε² écart à l'année
0,appartement,nb_ventes_n1,51.000,127.000,112.000,290.109,0.000,0.061,0.058
1,appartement,rotation_parc,0.429,0.700,0.714,229.297,0.000,0.048,0.039
2,appartement,rotation_parc_type,1.671,2.104,2.374,286.206,0.000,0.060,0.029
3,appartement,evol_ventes_n2_n1,7.248,9.756,9.491,5.305,0.070,0.001,0.002
4,appartement,evol_prix_n2_n1,5.877,2.310,3.251,67.376,0.000,0.014,0.037
5,maison,nb_ventes_n1,107.000,191.000,196.000,650.651,0.000,0.061,0.044
6,maison,rotation_parc,1.737,1.872,2.030,302.162,0.000,0.028,0.003
7,maison,rotation_parc_type,2.059,2.329,2.545,874.335,0.000,0.082,0.028
8,maison,evol_ventes_n2_n1,6.908,8.000,8.624,13.126,0.001,0.001,-0.000
9,maison,evol_prix_n2_n1,5.633,2.918,2.594,240.073,0.000,0.022,0.074


In [9]:
# Indicateurs qui séparent les groupes : ε² brut ≥ SEUIL_EPSILON2 pour au moins un type de bien
separe = kw.groupby("indicateur", sort=False)["ε² brut"].max() >= SEUIL_EPSILON2
RETENUS_EDA = [v for v in INDICATEURS if separe[v]]
print("Indicateurs qui séparent les groupes :", RETENUS_EDA)
print("Non retenus :", [v for v in INDICATEURS if not separe[v]])

Indicateurs qui séparent les groupes : ['nb_ventes_n1', 'rotation_parc', 'rotation_parc_type', 'evol_prix_n2_n1']
Non retenus : ['evol_ventes_n2_n1']


**Lecture**
- **Quatre indicateurs sur cinq séparent les groupes, avec de petits effets** (ε² de 0,01 à 0,08) : le volume de N-1 (0,061 pour les deux types de bien), la rotation du parc du même type (0,060 appartements, 0,082 maisons, le seul effet « moyen »), la rotation du parc total (0,048 et 0,028) et l'élan (0,014 et 0,022).
- **L'évolution des ventes ne sépare rien** (ε² = 0,001) : sur 2017-2022, il n'y a pas eu de retournement des ventes à annoncer. Elle n'entre pas dans les modèles.
- **Le volume** : les territoires en baisse ont moins de ventes (médiane de 51 contre 112 à 127 pour les appartements, 107 contre 191 à 196 pour les maisons). Avec une bande fixe, les petits marchés, plus bruités, tombent plus souvent dans une classe extrême : une partie de cet effet est le bruit des médianes, pas le marché.
- **L'élan sépare dans l'autre sens** : les territoires en baisse avaient l'élan le plus fort (médiane de 5,9 % contre 2,3 à 3,3 % pour les appartements). C'est un retour à la moyenne : une médiane montée haut en N-1 (vraie hausse ou bruit) a tendance à redescendre.
- **Écart à l'année** : l'élan sépare mieux à année égale (ε² de 0,037 et 0,074) : c'est un effet du territoire, le « où ». La rotation sépare moins bien (0,029 et 0,028) : une partie de son effet brut vient de la conjoncture.
- **Les p-values** sont toutes très petites, mais elles sont trop optimistes, puisqu'un même territoire revient chaque année : la décision repose sur ε².

### 2.3 Redondances entre indicateurs
Couples d'indicateurs dont |rho| de Spearman dépasse 0,7 : ils portent la même information. On n'en garde qu'un pour les modèles, celui qui sépare le mieux les groupes (ε² brut le plus grand).

In [10]:
couples = []
for tb, e in app.groupby("type"):
    rho = e[RETENUS_EDA].corr(method="spearman")
    for i, a in enumerate(RETENUS_EDA):
        for b in RETENUS_EDA[i + 1:]:
            couples.append({"type de bien": tb, "indicateur 1": a, "indicateur 2": b, "rho": rho.loc[a, b]})
couples = pd.DataFrame(couples)
print(couples.round(2).to_string(index=False))

# Un indicateur est écarté s'il est redondant (|rho| > 0,7 pour un type de bien) avec un indicateur qui sépare mieux
force = kw.groupby("indicateur")["ε² brut"].max()
ecartes = set()
for _, c in couples[couples["rho"].abs() > 0.7].iterrows():
    a, b = c["indicateur 1"], c["indicateur 2"]
    ecartes.add(a if force[a] < force[b] else b)
BLOC_H5 = [v for v in RETENUS_EDA if v not in ecartes]
print("Écartés pour redondance :", sorted(ecartes), "| bloc de H5 :", BLOC_H5)

type de bien       indicateur 1       indicateur 2  rho
 appartement       nb_ventes_n1      rotation_parc 0.81
 appartement       nb_ventes_n1 rotation_parc_type 0.43
 appartement       nb_ventes_n1    evol_prix_n2_n1 0.07
 appartement      rotation_parc rotation_parc_type 0.68
 appartement      rotation_parc    evol_prix_n2_n1 0.09
 appartement rotation_parc_type    evol_prix_n2_n1 0.18
      maison       nb_ventes_n1      rotation_parc 0.16
      maison       nb_ventes_n1 rotation_parc_type 0.42
      maison       nb_ventes_n1    evol_prix_n2_n1 0.08
      maison      rotation_parc rotation_parc_type 0.74
      maison      rotation_parc    evol_prix_n2_n1 0.13
      maison rotation_parc_type    evol_prix_n2_n1 0.22
Écartés pour redondance : ['rotation_parc'] | bloc de H5 : ['nb_ventes_n1', 'rotation_parc_type', 'evol_prix_n2_n1']


**Lecture** : la rotation du parc total est redondante avec le volume pour les appartements (rho = 0,81) et avec la rotation du même type pour les maisons (0,74). Elle sépare moins bien que les deux : elle est écartée. **Bloc de H5 : volume de N-1, rotation du parc du même type, élan.** Les trois sont peu liés entre eux (|rho| ≤ 0,43).

## 3. Fonctions du notebook
Le modèle de base est le **modèle pont** (`gabarit_pont.py`, décision H4) : Elastic Net sur le log du prix, prix N-1 + parc + vacance + densité, appris dès 2017, réglé par plis temporels. Les indicateurs de H5 y sont ajoutés comme un bloc (`bloc=...`), ceux de `LOG_H5` en log (`log_en_plus=...`).

`lancer_pont(nom, bloc)` fait, pour chaque année jugée et chaque type de bien : apprentissage du pont + bloc, prévision de l'année jugée, puis `evaluer` et `tester` de `commun.py` contre la naïve. Elle garde les prévisions (`previsions`, pour la comparaison au pont en 5.2) et les modèles ajustés (`ajustes`, pour les coefficients en 6.2). Relancer une cellule remplace les résultats du modèle.

On ne refait pas ici le Lasso, le gradient boosting ni le modèle hiérarchique du template : au tour 2, toutes les hypothèses (H5, H6, H7) partent du même pont, pour que leurs résultats se comparent.

In [11]:
resultats, tests, ajustes, previsions = [], [], {}, {}


def lancer_pont(nom, bloc=(), donnees=None):
    """Pont + bloc pour les deux fenêtres et les deux types de bien, puis evaluer et tester ; affiche le résultat."""
    nom = f"{HYPOTHESE} – {nom}"
    donnees = table if donnees is None else donnees
    resultats[:] = [r for r in resultats if r["modèle"] != nom]
    tests[:] = [t for t in tests if t["modèle"] != nom]
    modeles = {}
    prev = previsions_pont(donnees, bloc=list(bloc), log_en_plus=[c for c in bloc if c in LOG_H5], modeles=modeles)
    for annee, (app_f, ev) in fenetres(donnees, annee_debut=ANNEE_DEBUT).items():
        assert prev[annee].notna().all(), f"{nom} : prévisions manquantes"
        previsions[(nom, annee)] = prev[annee]
        evaluer(nom, ev, prev[annee], resultats)
        tester(nom, ev, prev[annee], tests)
    for cle, m in modeles.items():
        ajustes[(nom,) + cle] = m
    res = pd.DataFrame(resultats)
    res = res[res["modèle"] == nom][["type de bien", "année jugée", "MAE (€/m²)", "MAE naïve (€/m²)",
                                     "gain sur la naïve (%)", "bassins où le modèle gagne (%)"]]
    print(nom, f"({len(MODELE_COMPLET) + len(bloc)} variables)")
    print(res.round(1).to_string(index=False))
    return nom

## 4. Modèles
### 4.1 Référence : le modèle pont
Sur les lignes de H5. Contrôle : écart avec les MAE publiées par H4 (sur toutes les lignes du masque commun), qui doit rester petit.

In [12]:
PONT = lancer_pont("Pont")

H4_PUBLIE = {("appartement", 2023): 158.7, ("appartement", 2024): 187.8, ("maison", 2023): 130.4, ("maison", 2024): 180.2}
r = pd.DataFrame(resultats).query("modèle == @PONT")
r["MAE publiée H4 (€/m²)"] = [H4_PUBLIE[(t, a)] for t, a in zip(r["type de bien"], r["année jugée"])]
r[["type de bien", "année jugée", "MAE (€/m²)", "MAE publiée H4 (€/m²)"]].round(1)

H5 – Pont (12 variables)
type de bien  année jugée  MAE (€/m²)  MAE naïve (€/m²)  gain sur la naïve (%)  bassins où le modèle gagne (%)
 appartement         2023       159.9             166.0                    3.7                            54.8
      maison         2023       130.1             111.8                  -16.4                            39.9
 appartement         2024       189.4             147.5                  -28.4                            29.8
      maison         2024       179.7             129.2                  -39.1                            23.4


,type de bien,année jugée,MAE (€/m²),MAE publiée H4 (€/m²)
0,appartement,2023,159.9,158.7
1,maison,2023,130.1,130.4
2,appartement,2024,189.4,187.8
3,maison,2024,179.7,180.2


### 4.2 Pont + un indicateur à la fois
Chaque indicateur du bloc est ajouté seul au pont : on voit ce que chacun apporte, sans qu'un autre le masque.

In [13]:
UN_PAR_UN = {}
for v in BLOC_H5:
    UN_PAR_UN[v] = lancer_pont(f"Pont + {v}", [FORME_MODELE[v]])
    print()

H5 – Pont + nb_ventes_n1 (13 variables)
type de bien  année jugée  MAE (€/m²)  MAE naïve (€/m²)  gain sur la naïve (%)  bassins où le modèle gagne (%)
 appartement         2023       169.6             166.0                   -2.1                            50.9
      maison         2023       137.0             111.8                  -22.6                            38.2
 appartement         2024       185.0             147.5                  -25.4                            32.3
      maison         2024       173.3             129.2                  -34.2                            24.5

H5 – Pont + rotation_parc_type (13 variables)
type de bien  année jugée  MAE (€/m²)  MAE naïve (€/m²)  gain sur la naïve (%)  bassins où le modèle gagne (%)
 appartement         2023       163.9             166.0                    1.2                            50.8
      maison         2023       141.3             111.8                  -26.4                            36.9
 appartement         2024

### 4.3 Pont + bloc complet
Tous les indicateurs du bloc ensemble.

In [14]:
BLOC_MODELE = [FORME_MODELE[v] for v in BLOC_H5]
COMPLET = lancer_pont("Pont + bloc H5", BLOC_MODELE)

H5 – Pont + bloc H5 (15 variables)
type de bien  année jugée  MAE (€/m²)  MAE naïve (€/m²)  gain sur la naïve (%)  bassins où le modèle gagne (%)
 appartement         2023       162.4             166.0                    2.2                            54.0
      maison         2023       125.9             111.8                  -12.6                            43.1
 appartement         2024       188.4             147.5                  -27.7                            32.0
      maison         2024       173.5             129.2                  -34.3                            28.4


### 4.4 Le « quand » ou le « où » ? Part commune de l'année et écart du territoire
Chaque indicateur est séparé en deux : la **médiane de l'année** (même type de bien, tous territoires : c'est la conjoncture) et l'**écart du territoire** à cette médiane. Les deux ne demandent que des informations de N-1 et N-2, connues au moment de prévoir. On ajoute au pont l'une ou l'autre :
- si le gain vient de la part commune, H5 apporte surtout du « quand », comme l'emploi dans H3c ;
- s'il vient de l'écart, H5 distingue vraiment les territoires (le « où »).

La part commune n'a qu'une valeur par année : avec 6 ou 7 années d'apprentissage, le modèle n'en voit que 6 ou 7. C'est une lecture d'interprétation, pas un modèle à retenir tel quel.

In [15]:
annee_type = [table["annee"], table.index.get_level_values("type_bien")]
table_decomp = table.copy()
COMMUNE, LOCALE = [], []
for v in BLOC_MODELE:
    x = np.log(table[v]) if v in LOG_H5 else table[v]          # même échelle que dans le modèle
    mediane = x.groupby(annee_type).transform("median")
    table_decomp[v + "_annee"] = mediane
    table_decomp[v + "_ecart"] = x - mediane
    COMMUNE.append(v + "_annee")
    LOCALE.append(v + "_ecart")

PART_COMMUNE = lancer_pont("Pont + part commune de l'année", COMMUNE, donnees=table_decomp)
print()
ECART_LOCAL = lancer_pont("Pont + écart du territoire", LOCALE, donnees=table_decomp)

H5 – Pont + part commune de l'année (15 variables)
type de bien  année jugée  MAE (€/m²)  MAE naïve (€/m²)  gain sur la naïve (%)  bassins où le modèle gagne (%)
 appartement         2023       221.1             166.0                  -33.2                            39.3
      maison         2023       201.7             111.8                  -80.5                            25.2
 appartement         2024       172.8             147.5                  -17.1                            33.4
      maison         2024       143.1             129.2                  -10.8                            34.6

H5 – Pont + écart du territoire (15 variables)
type de bien  année jugée  MAE (€/m²)  MAE naïve (€/m²)  gain sur la naïve (%)  bassins où le modèle gagne (%)
 appartement         2023       178.5             166.0                   -7.5                            53.4
      maison         2023       131.5             111.8                  -17.6                            41.3
 appartement 

### 4.5 Sensibilité : l'année 2020 (confinement)
Les ventes de 2020 sont faussées par le confinement. Elles entrent dans les indicateurs des prix 2021 (ventes N-1 = 2020) et 2022 (ventes N-2 = 2020). On ajoute une indicatrice `confinement` (1 pour les prix 2021 et 2022, 0 sinon) **au pont et au pont + bloc**, et on compare les deux : le bloc garde-t-il son apport quand les années touchées ont leur propre décalage ? L'indicatrice vaut 0 sur les années jugées : elle ne sert qu'à l'apprentissage.

In [16]:
table_conf = table.copy()
table_conf["confinement"] = table_conf["annee"].isin([2021, 2022]).astype(float)
PONT_CONF = lancer_pont("Pont + indicatrice 2020", ["confinement"], donnees=table_conf)
print()
COMPLET_CONF = lancer_pont("Pont + indicatrice 2020 + bloc H5", ["confinement"] + BLOC_MODELE, donnees=table_conf)

H5 – Pont + indicatrice 2020 (13 variables)
type de bien  année jugée  MAE (€/m²)  MAE naïve (€/m²)  gain sur la naïve (%)  bassins où le modèle gagne (%)
 appartement         2023       155.9             166.0                    6.1                            58.1
      maison         2023       110.9             111.8                    0.7                            50.5
 appartement         2024       163.0             147.5                  -10.5                            36.7
      maison         2024       144.7             129.2                  -12.0                            36.0

H5 – Pont + indicatrice 2020 + bloc H5 (16 variables)
type de bien  année jugée  MAE (€/m²)  MAE naïve (€/m²)  gain sur la naïve (%)  bassins où le modèle gagne (%)
 appartement         2023       155.6             166.0                    6.3                            60.5
      maison         2023       110.1             111.8                    1.5                            51.5
 appartement 

## 5. Mesures et tests
Trois tableaux, du plus simple au plus exigeant :
1. **`res` : les erreurs.** MAE du modèle et de la naïve en €/m², gain sur la naïve en %, part des sous-bassins où le modèle gagne.
2. **`tst` : les tests contre la naïve.** Écart de MAE avec la naïve (négatif : le modèle fait mieux), IC 95 % du bootstrap par bassin de vie, p-value de Wilcoxon, rang-bisériale (de −1, mieux partout, à +1, moins bien partout).
3. **`decision` : la règle de décision.** Une année est réussie si l'IC est entièrement sous 0, la rang-bisériale négative et le gain d'au moins 3 %. Un modèle est retenu s'il réussit les deux années jugées.

In [17]:
res = pd.DataFrame(resultats)
res.round(2)

,modèle,type de bien,année jugée,MAE (€/m²),MAE en % du prix,MAE naïve (€/m²),gain sur la naïve (%),bassins où le modèle gagne (%)
0,H5 – Pont,appartement,2023,159.89,6.00,166.02,3.69,54.82
1,H5 – Pont,maison,2023,130.05,5.91,111.76,-16.36,39.91
2,H5 – Pont,appartement,2024,189.41,7.18,147.53,-28.39,29.76
3,H5 – Pont,maison,2024,179.68,7.93,129.19,-39.08,23.42
4,H5 – Pont + nb_ventes_n1,appartement,2023,169.57,6.28,166.02,-2.14,50.94
5,H5 – Pont + nb_ventes_n1,maison,2023,137.03,6.20,111.76,-22.61,38.22
6,H5 – Pont + nb_ventes_n1,appartement,2024,185.01,6.98,147.53,-25.41,32.32
7,H5 – Pont + nb_ventes_n1,maison,2024,173.31,7.70,129.19,-34.16,24.55
8,H5 – Pont + rotation_parc_type,appartement,2023,163.95,6.24,166.02,1.25,50.82
9,H5 – Pont + rotation_parc_type,maison,2023,141.29,6.48,111.76,-26.42,36.93


In [18]:
tst = pd.DataFrame(tests)
tst.round(3)

,modèle,type de bien,année jugée,écart de MAE avec la naïve (€/m²),IC 95 % bas (€/m²),IC 95 % haut (€/m²),p-value Wilcoxon,rang-bisériale,bassins de vie
0,H5 – Pont,appartement,2023,-6.126,-19.646,10.880,0.085,-0.068,716
1,H5 – Pont,maison,2023,18.288,13.003,23.207,0.000,0.253,1606
2,H5 – Pont,appartement,2024,41.877,33.160,48.551,0.000,0.479,689
3,H5 – Pont,maison,2024,50.491,46.726,54.367,0.000,0.675,1605
4,H5 – Pont + nb_ventes_n1,appartement,2023,3.551,-14.732,29.582,0.844,0.008,716
5,H5 – Pont + nb_ventes_n1,maison,2023,25.266,19.158,31.117,0.000,0.309,1606
6,H5 – Pont + nb_ventes_n1,appartement,2024,37.481,29.092,45.018,0.000,0.442,689
7,H5 – Pont + nb_ventes_n1,maison,2024,44.126,40.685,47.616,0.000,0.640,1605
8,H5 – Pont + rotation_parc_type,appartement,2023,-2.070,-14.766,12.597,0.965,-0.002,716
9,H5 – Pont + rotation_parc_type,maison,2023,29.524,23.526,35.533,0.000,0.329,1606


### 5.1 Règle de décision contre la naïve

In [19]:
def regle_decision(res, tst, comparaison="naïve"):
    """Années réussies et décision, par modèle et type de bien."""
    cles = ["modèle", "type de bien", "année jugée"]
    regle = res[cles + [f"gain sur la {comparaison} (%)"]].merge(
        tst[cles + ["IC 95 % haut (€/m²)", "rang-bisériale"]], on=cles)
    regle["année réussie"] = ((regle["IC 95 % haut (€/m²)"] < 0) & (regle["rang-bisériale"] < 0)
                              & (regle[f"gain sur la {comparaison} (%)"] >= SEUIL_GAIN))
    decision = regle.groupby(["modèle", "type de bien"], sort=False)["année réussie"].agg(["sum", "count"])
    decision["retenu"] = decision["sum"] > decision["count"] / 2
    return decision.rename(columns={"sum": "années réussies", "count": "années jugées"}), regle


decision_naive, regle_naive = regle_decision(res, tst)
decision_naive

années réussies  \
modèle                                 type de bien                    
H5 – Pont                              appartement                 0   
                                       maison                      0   
H5 – Pont + nb_ventes_n1               appartement                 0   
                                       maison                      0   
H5 – Pont + rotation_parc_type         appartement                 0   
                                       maison                      0   
H5 – Pont + evol_prix_n2_n1            appartement                 0   
                                       maison                      0   
H5 – Pont + bloc H5                    appartement                 0   
                                       maison                      0   
H5 – Pont + part commune de l'année    appartement                 0   
                                       maison                      0   
H5 – Pont + écart du territoire        appartement                 0   
                                       maison                      0   
H5 – Pont + indicatrice 2020           appartement                 1   
                                       maison                      0   
H5 – Pont + indicatrice 2020 + bloc H5 appartement                 1   
                                       maison                      0   

                                                     années jugées  retenu  
modèle                                 type de bien                         
H5 – Pont                              appartement               2   False  
                                       maison                    2   False  
H5 – Pont + nb_ventes_n1               appartement               2   False  
                                       maison                    2   False  
H5 – Pont + rotation_parc_type         appartement               2   False  
                                       maison                    2   False  
H5 – Pont + evol_prix_n2_n1            appartement               2   False  
                                       maison                    2   False  
H5 – Pont + bloc H5                    appartement               2   False  
                                       maison                    2   False  
H5 – Pont + part commune de l'année    appartement               2   False  
                                       maison                    2   False  
H5 – Pont + écart du territoire        appartement               2   False  
                                       maison                    2   False  
H5 – Pont + indicatrice 2020           appartement               2   False  
                                       maison                    2   False  
H5 – Pont + indicatrice 2020 + bloc H5 appartement               2   False  
                                       maison                    2   False

**Lecture** : comme dans H4, aucun modèle ne bat la naïve sur les deux années. En 2024, tous perdent, de 10 à 59 %. En 2023, le pont gagne 3,7 % pour les appartements, sans que l'écart soit net. Seul le pont + indicatrice 2020 réussit une année (appartements 2023 : IC haut −4,2 €/m², rang-bisériale −0,21, gain 6,1 %). Le problème du retournement n'est pas réglé par H5.

### 5.2 Le bloc apporte-t-il quelque chose au modèle pont ?
La question de H5 : mêmes mesures et même règle de décision, avec la prévision du **modèle pont** à la place de la naïve. `evaluer` et `tester` lisent la prévision de comparaison dans la colonne `prix_n1` : on leur donne une copie de l'année jugée où cette colonne contient la prévision du pont. Le pont + indicatrice 2020 est comparé au pont ; le pont + indicatrice 2020 + bloc, au pont avec la même indicatrice.

In [20]:
REFERENCE_DE = {n: PONT for n in dict.fromkeys(k[0] for k in previsions) if n != PONT}
REFERENCE_DE[COMPLET_CONF] = PONT_CONF          # même indicatrice des deux côtés

resultats_ref, tests_ref = [], []
for nom, ref in REFERENCE_DE.items():
    for annee, (app_f, ev) in FENETRES.items():
        ev_ref = ev.copy()
        ev_ref["prix_n1"] = previsions[(ref, annee)]
        evaluer(nom, ev_ref, previsions[(nom, annee)], resultats_ref)
        tester(nom, ev_ref, previsions[(nom, annee)], tests_ref)

res_ref = pd.DataFrame(resultats_ref).rename(columns=lambda c: c.replace("naïve", "référence"))
tst_ref = pd.DataFrame(tests_ref).rename(columns=lambda c: c.replace("naïve", "référence"))
display(res_ref[["modèle", "type de bien", "année jugée", "MAE (€/m²)", "MAE référence (€/m²)",
                 "gain sur la référence (%)", "bassins où le modèle gagne (%)"]].round(2))
display(tst_ref[["modèle", "type de bien", "année jugée", "écart de MAE avec la référence (€/m²)",
                 "IC 95 % bas (€/m²)", "IC 95 % haut (€/m²)", "p-value Wilcoxon", "rang-bisériale"]].round(3))

,modèle,type de bien,année jugée,MAE (€/m²),MAE référence (€/m²),gain sur la référence (%),bassins où le modèle gagne (%)
0,H5 – Pont + nb_ventes_n1,appartement,2023,169.57,159.89,-6.05,43.53
1,H5 – Pont + nb_ventes_n1,maison,2023,137.03,130.05,-5.37,36.82
2,H5 – Pont + nb_ventes_n1,appartement,2024,185.01,189.41,2.32,54.02
3,H5 – Pont + nb_ventes_n1,maison,2024,173.31,179.68,3.54,62.05
4,H5 – Pont + rotation_parc_type,appartement,2023,163.95,159.89,-2.54,44.59
5,H5 – Pont + rotation_parc_type,maison,2023,141.29,130.05,-8.64,36.31
6,H5 – Pont + rotation_parc_type,appartement,2024,181.21,189.41,4.33,57.07
7,H5 – Pont + rotation_parc_type,maison,2024,158.00,179.68,12.07,73.25
8,H5 – Pont + evol_prix_n2_n1,appartement,2023,159.30,159.89,0.37,56.24
9,H5 – Pont + evol_prix_n2_n1,maison,2023,115.18,130.05,11.43,66.72


,modèle,type de bien,année jugée,écart de MAE avec la référence (€/m²),IC 95 % bas (€/m²),IC 95 % haut (€/m²),p-value Wilcoxon,rang-bisériale
0,H5 – Pont + nb_ventes_n1,appartement,2023,9.677,2.802,19.334,0.000,0.168
1,H5 – Pont + nb_ventes_n1,maison,2023,6.978,4.561,9.897,0.000,0.341
2,H5 – Pont + nb_ventes_n1,appartement,2024,-4.396,-8.656,-0.121,0.000,-0.146
3,H5 – Pont + nb_ventes_n1,maison,2024,-6.365,-8.403,-4.652,0.000,-0.322
4,H5 – Pont + rotation_parc_type,appartement,2023,4.056,-0.277,8.446,0.001,0.132
5,H5 – Pont + rotation_parc_type,maison,2023,11.236,8.335,14.583,0.000,0.327
6,H5 – Pont + rotation_parc_type,appartement,2024,-8.194,-16.880,-1.567,0.000,-0.174
7,H5 – Pont + rotation_parc_type,maison,2024,-21.678,-24.410,-19.291,0.000,-0.621
8,H5 – Pont + evol_prix_n2_n1,appartement,2023,-0.586,-5.742,6.603,0.004,-0.113
9,H5 – Pont + evol_prix_n2_n1,maison,2023,-14.868,-17.132,-12.630,0.000,-0.426


#### Règle de décision contre le pont, 2023 lue à part
2023 est l'année du retournement : l'apprentissage (2017-2022) ne contient aucune baisse. 2024 est jugée avec 2023 dans l'apprentissage. On lit donc chaque année séparément avant la décision sur les deux années.

In [21]:
decision_pont, regle_pont = regle_decision(res_ref, tst_ref, comparaison="référence")
annees_lues = (regle_pont.pivot_table(index=["modèle", "type de bien"], columns="année jugée",
                                      values=["gain sur la référence (%)", "IC 95 % haut (€/m²)", "année réussie"],
                                      aggfunc="first", sort=False))
display(annees_lues.round(2))
decision_pont

gain sur la référence (%)  \
année jugée                                                              2023   
modèle                                 type de bien                             
H5 – Pont + nb_ventes_n1               appartement                      -6.05   
                                       maison                           -5.37   
H5 – Pont + rotation_parc_type         appartement                      -2.54   
                                       maison                           -8.64   
H5 – Pont + evol_prix_n2_n1            appartement                       0.37   
                                       maison                           11.43   
H5 – Pont + bloc H5                    appartement                      -1.54   
                                       maison                            3.22   
H5 – Pont + part commune de l'année    appartement                     -38.27   
                                       maison                          -55.13   
H5 – Pont + écart du territoire        appartement                     -11.66   
                                       maison                           -1.08   
H5 – Pont + indicatrice 2020           appartement                       2.49   
                                       maison                           14.69   
H5 – Pont + indicatrice 2020 + bloc H5 appartement                       0.21   
                                       maison                            0.73   

                                                            \
année jugée                                           2024   
modèle                                 type de bien          
H5 – Pont + nb_ventes_n1               appartement    2.32   
                                       maison         3.54   
H5 – Pont + rotation_parc_type         appartement    4.33   
                                       maison        12.07   
H5 – Pont + evol_prix_n2_n1            appartement   -3.92   
                                       maison       -14.04   
H5 – Pont + bloc H5                    appartement    0.54   
                                       maison         3.45   
H5 – Pont + part commune de l'année    appartement    8.77   
                                       maison        20.37   
H5 – Pont + écart du territoire        appartement   -1.44   
                                       maison        -2.01   
H5 – Pont + indicatrice 2020           appartement   13.97   
                                       maison        19.49   
H5 – Pont + indicatrice 2020 + bloc H5 appartement   -1.41   
                                       maison        -6.58   

                                                    IC 95 % haut (€/m²)  \
année jugée                                                        2023   
modèle                                 type de bien                       
H5 – Pont + nb_ventes_n1               appartement                19.33   
                                       maison                      9.90   
H5 – Pont + rotation_parc_type         appartement                 8.45   
                                       maison                     14.58   
H5 – Pont + evol_prix_n2_n1            appartement                 6.60   
                                       maison                    -12.63   
H5 – Pont + bloc H5                    appartement                 9.07   
                                       maison                     -0.59   
H5 – Pont + part commune de l'année    appartement                87.49   
                                       maison                     75.86   
H5 – Pont + écart du territoire        appartement                48.42   
                                       maison                      4.46   
H5 – Pont + indicatrice 2020           appartement                 5.27   
                                       maison                    -14.68   
H5 – Pont + indicatrice 2020 + bloc H5 appartement        

années réussies  \
modèle                                 type de bien                    
H5 – Pont + nb_ventes_n1               appartement                 0   
                                       maison                      1   
H5 – Pont + rotation_parc_type         appartement                 1   
                                       maison                      1   
H5 – Pont + evol_prix_n2_n1            appartement                 0   
                                       maison                      1   
H5 – Pont + bloc H5                    appartement                 0   
                                       maison                      2   
H5 – Pont + part commune de l'année    appartement                 1   
                                       maison                      1   
H5 – Pont + écart du territoire        appartement                 0   
                                       maison                      0   
H5 – Pont + indicatrice 2020           appartement                 1   
                                       maison                      2   
H5 – Pont + indicatrice 2020 + bloc H5 appartement                 0   
                                       maison                      0   

                                                     années jugées  retenu  
modèle                                 type de bien                         
H5 – Pont + nb_ventes_n1               appartement               2   False  
                                       maison                    2   False  
H5 – Pont + rotation_parc_type         appartement               2   False  
                                       maison                    2   False  
H5 – Pont + evol_prix_n2_n1            appartement               2   False  
                                       maison                    2   False  
H5 – Pont + bloc H5                    appartement               2   False  
                                       maison                    2    True  
H5 – Pont + part commune de l'année    appartement               2   False  
                                       maison                    2   False  
H5 – Pont + écart du territoire        appartement               2   False  
                                       maison                    2   False  
H5 – Pont + indicatrice 2020           appartement               2   False  
                                       maison                    2    True  
H5 – Pont + indicatrice 2020 + bloc H5 appartement               2   False  
                                       maison                    2   False

**Lecture**
- **Seuls, les indicateurs ne sont pas retenus, et ils s'opposent d'une année à l'autre.** Le volume et la rotation perdent en 2023 (de −2,5 à −8,6 %) et gagnent en 2024 (de +2,3 à +12,1 %, la rotation pour les maisons). L'élan fait l'inverse : +11,4 % pour les maisons en 2023, −14,0 % en 2024.
- **Le bloc complet est retenu pour les maisons, de justesse** : +3,2 % en 2023 (IC haut −0,6 €/m², rang-bisériale −0,08) et +3,5 % en 2024 (IC haut −3,1, rang-bisériale −0,14), à peine au-dessus du seuil de 3 %. Il n'est pas retenu pour les appartements (−1,5 % puis +0,5 %).
- **2023 lue à part** : la seule vraie aide de 2023 vient de l'élan pour les maisons (+11,4 %, IC entièrement sous 0). Son coefficient est négatif : un territoire qui a beaucoup monté est prévu moins haut, ce qui freine le prolongement de la hausse l'année où elle s'arrête. En 2024, l'élan est retombé (+0,8 % en médiane, P9), ce frein disparaît et l'élan dessert le modèle.
- **Part commune ou écart du territoire** : la part commune de l'année est très instable (−38 et −55 % en 2023, +9 et +20 % en 2024) ; elle n'a que 6 ou 7 valeurs en apprentissage. L'écart du territoire seul n'apporte rien (de −12 à −1 %). Le gain du bloc ne se ramène à aucune des deux parts seule.
- **Le contrôle de 2020 change la lecture.** Le pont + indicatrice 2020 bat nettement le pont : +2,5 et +14,0 % pour les appartements, **+14,7 et +19,5 % pour les maisons (retenu les deux années)**. Et une fois cette indicatrice dans le modèle, le bloc H5 n'apporte plus rien : 0 année réussie, −6,6 % pour les maisons en 2024. **Le petit gain du bloc pour les maisons est donc fragile** : il disparaît dès que les prix 2021 et 2022 ont leur propre niveau.

## 6. Interprétation
### 6.1 Contexte : le marché s'est-il retourné ?
Évolution du prix d'une année sur l'autre (prix / prix N-1 − 1), médiane et part des classes avec la bande retenue, par année et type de bien, jusqu'à 2024. L'année 2025, réservée au test final, n'est pas regardée.

In [22]:
jusqua_2024 = table[(table["annee"] >= ANNEE_DEBUT) & (table["annee"] <= commun.ANNEE_VALIDATION)]
evolution = (jusqua_2024[CIBLE] / jusqua_2024["prix_n1"] - 1) * 100
cle_ctx = [jusqua_2024["annee"], jusqua_2024.index.get_level_values("type_bien")]
contexte = pd.DataFrame({"évolution médiane (%)": evolution.groupby(cle_ctx).median(),
                         "baisse (%)": (evolution < -BANDE_STABLE).groupby(cle_ctx).mean() * 100,
                         "hausse (%)": (evolution > BANDE_STABLE).groupby(cle_ctx).mean() * 100}).round(1)
contexte.unstack()

évolution médiane (%)         baisse (%)         hausse (%)       
type_bien           appartement maison appartement maison appartement maison
annee                                                                       
2017                        2.2    2.0        19.5   19.1        43.9   40.9
2018                        1.4    2.1        24.5   17.8        38.0   42.9
2019                        3.0    2.3        15.8   18.5        49.5   43.6
2020                        5.1    4.7        10.8   10.3        63.4   62.3
2021                        7.6    7.7         6.1    5.1        77.2   80.2
2022                        8.3    9.1         4.6    3.0        80.0   85.7
2023                        2.8    0.8        20.6   24.2        47.9   34.8
2024                       -1.5   -2.9        40.1   49.2        25.6   17.6

### 6.2 Poids des indicateurs dans le pont + bloc
Coefficients standardisés de l'Elastic Net (variables mises à l'échelle, cible en log) : le signe dit le sens de l'effet à variables égales ; 0 : l'Elastic Net a écarté la variable. Par fenêtre et type de bien, avec le coefficient du prix N-1 pour l'ordre de grandeur.

In [23]:
coefs = pd.DataFrame({(a, tb): m.coefs_[["prix_n1"] + BLOC_MODELE]
                      for (nom, a, tb), m in ajustes.items() if nom == COMPLET})
coefs.columns.names = ["année jugée", "type de bien"]
coefs.round(4)

année jugée               2023                2024        
type de bien       appartement  maison appartement  maison
prix_n1                 0.5146  0.4491      0.5172  0.4515
nb_ventes_n1            0.0000 -0.0300      0.0000 -0.0266
rotation_parc_type      0.0197  0.0387      0.0173  0.0332
evol_prix_n2_n1        -0.0130 -0.0247     -0.0151 -0.0265

**Lecture** : les trois indicateurs pèsent peu à côté du prix N-1 (coefficients standardisés de 0,01 à 0,04 en valeur absolue, contre 0,45 à 0,52 pour le prix N-1). L'élan a un coefficient négatif partout (retour à la moyenne, section 2.2). La rotation du même type est positive : à prix N-1 égal, un marché qui tourne beaucoup est prévu plus cher. Le volume est écarté par l'Elastic Net pour les appartements (0) et légèrement négatif pour les maisons. Les coefficients sont stables entre les deux fenêtres.

### 6.3 Analyse des erreurs selon le contrôle qualité
L'erreur du pont + bloc est-elle plus grande là où les données sont estimées ou complétées ? Pour chaque variable de contrôle qualité : groupe « part nulle », puis quartiles parmi les parts non nulles. Par groupe : erreur moyenne en €/m² et en % du prix, et erreur signée (positive : le modèle surestime). Année jugée 2024, les deux types de bien.

In [24]:
CONTROLE = ["part_indicateurs_estimes", "part_logements_taux_impute", "part_menages_couverts"]
ev_24 = FENETRES[2024][1]
erreur = previsions[(COMPLET, 2024)] - ev_24[CIBLE]
erreur_pct = erreur.abs() / ev_24[CIBLE] * 100

for v in CONTROLE:
    x = ev_24[v]
    groupe = pd.Series("part nulle", index=x.index)
    non_nul = x > 0
    if non_nul.sum() >= 4:
        groupe[non_nul] = pd.qcut(x[non_nul], 4, labels=["Q1", "Q2", "Q3", "Q4"], duplicates="drop").astype(str)
    tableau = pd.DataFrame({"MAE (€/m²)": erreur.abs(), "erreur en % du prix": erreur_pct,
                            "erreur signée (€/m²)": erreur}).groupby(groupe).mean()
    tableau["lignes"] = groupe.value_counts()
    rho, p = stats.spearmanr(x, erreur.abs())
    print(f"\n{v} : rho avec l'erreur en €/m² = {rho:.2f} (p = {p:.3f})")
    print(tableau.round(1).to_string())


part_indicateurs_estimes : rho avec l'erreur en €/m² = -0.16 (p = 0.000)
            MAE (€/m²)  erreur en % du prix  erreur signée (€/m²)  lignes
Q1               213.1                  7.3                 195.6     633
Q2               160.2                  7.4                 120.6     632
Q3               150.6                  7.5                 106.4     631
Q4               160.9                  7.5                 125.5     632
part nulle       437.3                  8.7                 434.7      68

part_logements_taux_impute : rho avec l'erreur en €/m² = -0.32 (p = 0.000)
            MAE (€/m²)  erreur en % du prix  erreur signée (€/m²)  lignes
Q1               192.1                  7.3                 159.7     479
Q2               144.0                  7.1                 107.6     479
Q3               125.9                  7.0                  93.8     478
Q4               124.3                  7.8                  88.8     479
part nulle       267.0              

### 6.4 Où le bloc aide-t-il ? Gain sur le pont par taille de marché
Écart d'erreur absolue (pont + bloc moins pont) par quintile de ventes N-1, et biais (erreur signée moyenne) des deux modèles. Négatif : le bloc fait mieux. Le bruit des médianes étant plus grand dans les petits marchés (section 2.1), un indicateur de volume pourrait surtout aider à y corriger le prix N-1.

In [25]:
lignes_taille = []
for annee, (app_f, ev) in FENETRES.items():
    for tb in ["appartement", "maison"]:
        e = ev[ev.index.get_level_values("type_bien") == tb]
        err_b = previsions[(COMPLET, annee)].loc[e.index] - e[CIBLE]
        err_p = previsions[(PONT, annee)].loc[e.index] - e[CIBLE]
        q = pd.qcut(e["nb_ventes_n1"].rank(method="first"), 5, labels=["Q1 (petits)", "Q2", "Q3", "Q4", "Q5 (grands)"])
        for k, idx in e.groupby(q, observed=True).groups.items():
            lignes_taille.append({"année jugée": annee, "type de bien": tb, "quintile de ventes N-1": k,
                                  "écart d'erreur bloc − pont (€/m²)": (err_b.loc[idx].abs() - err_p.loc[idx].abs()).mean(),
                                  "biais pont (€/m²)": err_p.loc[idx].mean(), "biais pont + bloc (€/m²)": err_b.loc[idx].mean()})
par_taille = pd.DataFrame(lignes_taille).set_index(["année jugée", "type de bien", "quintile de ventes N-1"])
par_taille.round(1)

écart d'erreur bloc − pont (€/m²)  \
année jugée type de bien quintile de ventes N-1                                      
2023        appartement  Q1 (petits)                                         -10.0   
                         Q2                                                   -2.0   
                         Q3                                                    5.5   
                         Q4                                                    1.5   
                         Q5 (grands)                                          17.3   
            maison       Q1 (petits)                                         -12.7   
                         Q2                                                  -10.4   
                         Q3                                                   -3.6   
                         Q4                                                    0.1   
                         Q5 (grands)                                           5.6   
2024        appartement  Q1 (petits)                                         -13.2   
                         Q2                                                   -6.3   
                         Q3                                                   -0.7   
                         Q4                                                    0.8   
                         Q5 (grands)                                          14.2   
            maison       Q1 (petits)                                         -13.4   
                         Q2                                                   -6.1   
                         Q3                                                   -4.7   
                         Q4                                                   -3.7   
                         Q5 (grands)                                          -3.1   

                                                 biais pont (€/m²)  \
année jugée type de bien quintile de ventes N-1                      
2023        appartement  Q1 (petits)                          46.7   
                         Q2                                   47.2   
                         Q3                                   15.5   
                         Q4                                  -12.0   
                         Q5 (grands)                         130.1   
            maison       Q1 (petits)                          58.8   
                         Q2                                   61.6   
                         Q3                                   63.8   
                         Q4                                   85.0   
                         Q5 (grands)                          88.1   
2024        appartement  Q1 (petits)                         104.7   
                         Q2                                   79.9   
                         Q3                                  137.3   
                         Q4                                  130.7   
                         Q5 (grands)                         240.3   
            maison       Q1 (petits)                         164.9   
                         Q2                                  122.4   
                         Q3                                  153.7   
                         Q4                                  146.1   
                         Q5 (grands)                         162.8   

                                                 biais pont + bloc (€/m²)  
année jugée type de bien quintile de ventes N-1                            
2023        appartement  Q1 (petits)                                 23.6  
                         Q2                                          49.4  
                         Q3                                          27.2  
                         Q4                                          -6.8  
                         Q5 (grands)                                159.4  
            maison       Q1 (petits)                                 32.2  
       

**Lecture**
- **Le bloc aide dans les petits marchés** : dans le premier quintile de ventes, l'erreur baisse de 10 à 13 €/m² chaque année, pour les deux types de bien. Pour les maisons de 2023, le biais y passe de +59 à +32 €/m². C'est là que la médiane N-1 est la plus bruitée (section 2.1) : le volume et l'élan aident à corriger un prix N-1 peu fiable.
- **Pour les appartements, il dessert les grands marchés** (+17 €/m² en 2023, +14 en 2024 dans le dernier quintile), ce qui annule le gain. Pour les maisons, l'effet dans les grands marchés est presque nul, d'où le petit gain d'ensemble.

### 6.5 L'élan annonce-t-il le retournement ? L'année 2023 lue à part
En 2023, les modèles appris sur 2017-2022 n'ont jamais vu de baisse. Si l'élan et le volume annoncent le sens de l'évolution, le pont + bloc devrait moins prolonger la hausse que le pont : biais (erreur signée moyenne) des deux modèles, par année jugée et type de bien.

In [26]:
biais = []
for nom in [PONT, COMPLET, PART_COMMUNE, ECART_LOCAL, PONT_CONF, COMPLET_CONF]:
    for annee, (app_f, ev) in FENETRES.items():
        err = previsions[(nom, annee)] - ev[CIBLE]
        for tb, idx in ev.groupby(level="type_bien").groups.items():
            biais.append({"modèle": nom, "année jugée": annee, "type de bien": tb, "biais (€/m²)": err.loc[idx].mean()})
pd.DataFrame(biais).pivot_table(index="modèle", columns=["année jugée", "type de bien"], values="biais (€/m²)", sort=False).round(1)

année jugée                                   2023               2024       
type de bien                           appartement maison appartement maison
modèle                                                                      
H5 – Pont                                     45.5   71.5       138.6  150.0
H5 – Pont + bloc H5                           50.6   61.7       137.9  148.1
H5 – Pont + part commune de l'année          169.7  176.2       115.6   96.6
H5 – Pont + écart du territoire               69.0   79.1       143.7  158.9
H5 – Pont + indicatrice 2020                 -20.0   -0.8        91.9   98.2
H5 – Pont + indicatrice 2020 + bloc H5       -25.7  -33.0        96.4  120.6

**Lecture**
- **Le bloc H5 n'annonce pas le retournement** : le biais du pont reste positif et presque inchangé avec le bloc (de +46 à +72 €/m² en 2023, de +138 à +150 en 2024). Le modèle prolonge toujours la hausse.
- **La part commune seule déplace le biais sans le réduire** : +170 à +176 €/m² en 2023, puis +97 à +116 en 2024.
- **L'indicatrice 2020 retire le biais de 2023** (−20 et −1 €/m²) et environ un tiers de celui de 2024 (+92 et +98 contre +139 et +150). Les prix 2021 et 2022 sont à la fois les années touchées par le confinement et le sommet de la hausse (+7,6 à +9,1 % en médiane, section 6.1). Leur donner leur propre niveau empêche le modèle de prolonger ce sommet. **L'effet n'est donc pas attribuable au confinement : c'est un effet « quand »**, à transmettre à l'équipe (H4, H6, H7).

## 7. Conclusion
**Question de H5** : par territoire, le volume de N-1, son évolution et l'élan du prix améliorent-ils la prévision du prix N ?

1. **Bande « stable » pour l'équipe : fixe, ±3 %** (section 2.1). C'est le bruit des grands marchés en années calmes. La bande dépendante du nombre de ventes rendait le classement plus dépendant de la taille du marché (V de Cramér jusqu'à 0,30 contre 0,16 à 0,18). Elle est donc écartée, contrairement à ce qui était proposé avant l'analyse. Pour H6 : `BANDE_STABLE = 3` et `classer`.
2. **Analyse préalable** (2.2, 2.3) : le volume de N-1, la rotation du parc du même type et l'élan séparent hausse, stable et baisse, avec de petits effets (ε² de 0,01 à 0,08). L'évolution des ventes ne sépare rien sur 2017-2022. Une partie de l'effet du volume vient du bruit des petits marchés ; celui de l'élan est un retour à la moyenne, plus net à année égale.
3. **Contre la naïve** (5.1) : aucun modèle n'est retenu. Le problème du retournement reste entier.
4. **Contre le modèle pont** (5.2) : le bloc H5 est **retenu pour les maisons, de justesse** (+3,2 % et +3,5 %, intervalles sous 0), et **pas pour les appartements**. Aucun indicateur seul n'est retenu : le volume et la rotation aident en 2024 et desservent en 2023, l'élan fait l'inverse.
5. **Mais le gain est fragile** : il disparaît quand on donne aux prix 2021-2022 leur propre niveau (indicatrice 2020). Le bloc ne réduit pas le biais du pont (6.5). Il aide surtout dans les petits marchés, en corrigeant un prix N-1 bruité (6.4).
6. **Découverte à partager** : l'indicatrice des prix 2021-2022 bat nettement le pont (+14,7 et +19,5 % pour les maisons, retenue ; +2,5 et +14,0 % pour les appartements). Elle retire le biais de 2023 et un tiers de celui de 2024. Ces années sont aussi le sommet de la hausse : c'est un effet « quand », à discuter avec Christian (pont) et pour H6-H7.

**Décision proposée** : ne pas ajouter le bloc H5 au pont tel quel. Il passe la règle pour les maisons, mais son gain est proche du seuil, instable indicateur par indicateur et absorbé par l'indicatrice des années 2021-2022. À rediscuter si l'équipe ajoute cette indicatrice au pont.

**Limites**
- Deux années jugées seulement, toutes deux pendant le retournement ; l'apprentissage 2017-2022 ne contient aucune baisse des ventes.
- 312 lignes retirées (moins de 20 ventes en N-2), surtout des appartements : le pont est jugé ici sur un peu moins de lignes que dans H4 (MAE à 1,6 €/m² près).
- Classes en partie dues au bruit des médianes ; p-values de Kruskal-Wallis trop optimistes (territoires répétés).
- Indicatrice « 2020 » confondue avec le sommet de la hausse : ce n'est pas un effet du confinement.
- Photos fixes (recensement 2022, LOVAC, densité) appliquées aux prix 2017-2019, comme dans H4.

Le test 2025 n'est pas ouvert.